In [ ]:
import subprocess
import re
import numpy as np

def parse_gradient(output, label):
    """Extract the vector from 'label = [ 1.0e+00, 2.0e+00, ... ]' (1D)."""
    m = re.search(rf"{re.escape(label)}\s*=\s*\[(.*?)\]", output, re.DOTALL)
    assert m, f"could not find '{label} = [...]' in output:\n{output}"
    body = m.group(1)
    return np.array([float(tok) for tok in body.split(",") if tok.strip()])

def parse_positions(output, label):
    """Extract '(a, b, c), (d, e, f), ...' from 'label = [ ... ]' into an (M,3) array."""
    m = re.search(rf"{re.escape(label)}\s*=\s*\[(.*?)\]", output, re.DOTALL)
    assert m, f"could not find '{label} = [...]' in output:\n{output}"
    body = m.group(1)
    # grab every (x, y, z) triple
    triples = re.findall(r"\(([^)]*)\)", body)
    rows = [[float(v) for v in t.split(",")] for t in triples]
    return np.array(rows)

def check_match(name, a, b, tol=1e-6):
    """Assert two arrays match within tol; print a one-line verdict."""
    assert a.shape == b.shape, f"{name}: shape mismatch {a.shape} vs {b.shape}"
    max_abs = np.max(np.abs(a - b))
    ok = max_abs < tol
    status = "OK " if ok else "FAIL"
    print(f"  [{status}] {name}: max abs diff = {max_abs:.3e}  (tol {tol:.0e})")
    return ok

def compare_vectors(p1, p2, label1="p1", label2="p2"):
    p1 = np.asarray(p1, dtype=float).ravel()
    p2 = np.asarray(p2, dtype=float).ravel()
    assert p1.shape == p2.shape, f"shape mismatch: {p1.shape} vs {p2.shape}"
    diff = p1 - p2

    print(f"{'i':>3} {label1:>16} {label2:>16} {'diff':>16} "
          f"{'rel_err':>12} {'ratio p2/p1':>14}")
    for i in range(len(p1)):
        d   = diff[i]
        rel = abs(d) / max(abs(p1[i]), abs(p2[i]), 1e-30)
        ratio = p2[i] / p1[i] if abs(p1[i]) > 1e-30 else float("nan")
        print(f"{i:>3} {p1[i]:16.6e} {p2[i]:16.6e} {d:+16.6e} "
              f"{rel:12.4e} {ratio:14.6f}")

    print()
    print(f"max abs diff:     {np.max(np.abs(diff)):.6e}")
    print(f"mean abs diff:    {np.mean(np.abs(diff)):.6e}")
    print(f"max relative err: {np.max(np.abs(diff) / np.maximum(np.abs(p1), 1e-30)):.6e}")
    cos = np.dot(p1, p2) / (np.linalg.norm(p1) * np.linalg.norm(p2))
    print(f"\ncosine similarity:   {cos:.10f}")
    print(f"ratio of norms |{label2}|/|{label1}|: {np.linalg.norm(p2) / np.linalg.norm(p1):.10f}")
    valid = np.abs(p1) > 1e-30
    if np.any(valid):
        ratios = p2[valid] / p1[valid]
        print(f"element-wise ratio: mean={ratios.mean():.6f}, "
              f"std={ratios.std():.6f}, min={ratios.min():.6f}, max={ratios.max():.6f}")

def run(cmd, cwd=None):
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"--- command failed: {cmd} ---")
        print("STDOUT:\n", result.stdout)
        print("STDERR:\n", result.stderr)
        raise RuntimeError(f"{cmd} exited with code {result.returncode}")
    return result.stdout

CPP_EXE    = r"C:\Users\Workstation\borsa_verona\DiffXPBD\build\bin\Release\xpbd.exe"
JAX_SCRIPT = r"C:\Users\Workstation\borsa_verona\DiffXPBD\jax_impl.py"

In [ ]:
print("Running C++ ...")
cpp_out = run([CPP_EXE])
print("Running JAX ...")
jax_out = run(["python", JAX_SCRIPT])

# --- forward check: positions must match ---
print("\n" + "=" * 60)
print("FORWARD CHECK (positions should match)")
print("=" * 60)

cpp_final = parse_positions(cpp_out, "pos_final")
jax_final = parse_positions(jax_out, "pos_final")
cpp_guess = parse_positions(cpp_out, "pos_guess")
jax_guess = parse_positions(jax_out, "pos_guess")

POS_TOL = 1e-6
ok_final = check_match("target final positions", cpp_final, jax_final, POS_TOL)
ok_guess = check_match("guess final positions",  cpp_guess, jax_guess, POS_TOL)

if not (ok_final and ok_guess):
    print("\n  WARNING: forward sims diverge — gradient comparison may be meaningless.")

# --- backward check: gradient comparison ---
print("\n" + "=" * 60)
print("GRADIENT COMPARISON")
print("=" * 60)

cpp_grad = parse_gradient(cpp_out, "dL_dalpha")
jax_grad = parse_gradient(jax_out, "dL_dalpha")
compare_vectors(cpp_grad, jax_grad, label1="C++", label2="JAX")

In [ ]:
# --- d loss / d x0 comparison ---
print("Running C++ ...")
cpp_out = run([CPP_EXE])
print("Running JAX ...")
jax_out = run(["python", JAX_SCRIPT])

# Optional: also check that the loss matches before trusting the gradient.
def parse_loss(output):
    m = re.search(r"loss:\s*([-\d.eE+]+)", output)
    assert m, f"could not find 'loss: ...' in output"
    return float(m.group(1))

cpp_loss = parse_loss(cpp_out)
jax_loss = parse_loss(jax_out)
print("\n" + "=" * 60)
print("LOSS CHECK")
print("=" * 60)
print(f"  C++ loss = {cpp_loss:.8e}")
print(f"  JAX loss = {jax_loss:.8e}")
print(f"  rel diff = {abs(cpp_loss - jax_loss) / max(abs(jax_loss), 1e-30):.3e}")

# --- gradient ---
print("\n" + "=" * 60)
print("dL/dx0 COMPARISON")
print("=" * 60)

cpp_grad = parse_gradient(cpp_out, "dL_dx0")
jax_grad = parse_gradient(jax_out, "dL_dx0")
compare_vectors(cpp_grad, jax_grad, label1="C++", label2="JAX")